# Generating Text with a Trained GPT

We've done the hard work: we've prepared data, built a GPT model, and orchestrated a training loop. Now for the payoff! By the end of this notebook, you will be able to load a trained model, provide it with a starting prompt, and watch it generate new text one token at a time. This is the magic of generative AI, and you'll understand exactly how it works under the hood.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import textwrap

# Set a manual seed for reproducibility
torch.manual_seed(42)

### The Core Idea: Autocomplete on Steroids

How does a model "write" text? The process is surprisingly simple and elegant. It's a lot like the autocomplete on your phone, but far more powerful. Your phone looks at the last few words you typed and suggests the most likely *next word*. A GPT model does the same thing, but it looks at a sequence of *tokens* (which can be words, or parts of words) and predicts the most likely *next token*.

This process is called **autoregressive sampling**. Let's break that down:
- **AutoRegressive**: "Auto" means self, and "regressive" means looking at the past. The model's future output depends on its own past output.
- **Sampling**: Instead of always picking the single most probable next token (which would be deterministic and boring), we *sample* from the distribution of probabilities the model provides. This introduces creativity and variation.

Here's the step-by-step loop of how it works:

1.  **Start with a prompt**: We give the model an initial sequence of tokens, like "To be, or not to".
2.  **Predict**: The model takes this sequence and predicts the probabilities for every possible next token in its vocabulary. The token " be" will likely have a very high probability.
3.  **Sample**: We pick one token based on these probabilities. Let's say we sample " be".
4.  **Append**: We append our new token to the sequence. Our sequence is now "To be, or not to be".
5.  **Repeat**: We take this new, longer sequence and feed it back into the model to generate the *next* token. We continue this loop until we've generated enough text.

It's a simple feedback loop: the model's output becomes its next input.

In [ ]:
# A mock tokenizer for character-level encoding/decoding
class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.char_to_int = {ch: i for i, ch in enumerate(self.chars)}
        self.int_to_char = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, string):
        return [self.char_to_int[c] for c in string]

    def decode(self, int_list):
        return ''.join([self.int_to_char[i] for i in int_list])

# A minimal mock GPT model. It doesn't need the full transformer architecture.
# The key is that its forward() method returns logits of shape (batch, sequence, vocab_size).
class MockGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # A real GPT has many complex layers. We just need the input/output shape to match.
        self.embedding = nn.Embedding(vocab_size, 16) # vocab_size -> embedding_dim
        self.linear = nn.Linear(16, vocab_size)       # embedding_dim -> vocab_size

    def forward(self, idx):
        # A real GPT would have transformer blocks here.
        # We'll just pass the embeddings to a linear layer to get the right shape.
        x = self.embedding(idx) # (batch, sequence_length) -> (batch, sequence_length, embedding_dim)
        logits = self.linear(x) # (batch, sequence_length, embedding_dim) -> (batch, sequence_length, vocab_size)
        return logits

def generate(model, prompt_indices, max_new_tokens):
    """
    Takes a model, a starting sequence of indices (the prompt),
    and generates up to max_new_tokens additional tokens.
    """
    # Our prompt_indices is a list, but the model expects a tensor of shape (batch, sequence).
    # So we add a batch dimension of 1.
    idx = torch.tensor(prompt_indices, dtype=torch.long).unsqueeze(0)

    for _ in range(max_new_tokens):
        # Get the model's predictions (logits).
        logits = model(idx)
        # We only care about the prediction for the very last token in the sequence.
        last_token_logits = logits[:, -1, :]
        # Convert logits to probabilities using softmax.
        probs = F.softmax(last_token_logits, dim=-1)
        # Sample the next token's index from the probability distribution.
        idx_next = torch.multinomial(probs, num_samples=1)
        # Append the sampled token index to our running sequence.
        idx = torch.cat((idx, idx_next), dim=1)

    # The final sequence is in a tensor, so we extract it and convert to a list.
    return idx[0].tolist()

### Walking Through the Code

Let's break down our `generate` function line by line. This is the heart of text generation.

1.  **`idx = torch.tensor(prompt_indices, ...).unsqueeze(0)`**
    Our model is designed to work with *batches* of sequences for efficient training. Even though we are only generating one sequence, we must still wrap our input `prompt_indices` in a batch. `unsqueeze(0)` adds a new dimension at the beginning, changing the shape from `(sequence_length)` to `(1, sequence_length)`.

2.  **`for _ in range(max_new_tokens):`**
    This is the main generation loop. We'll repeat the predict-sample-append cycle once for each new token we want to create.

3.  **`logits = model(idx)`**
    We feed the entire current sequence of tokens (`idx`) to the model. The model returns `logits`, which are raw, unnormalized scores for every possible token in the vocabulary, at *every position* in the input sequence. The shape of `logits` is `(batch_size, sequence_length, vocab_size)`.

4.  **`last_token_logits = logits[:, -1, :]`**
    This is a critical step. The model predicts what comes next for every token in the input. For the prompt "To be", it predicts what might come after "To" AND what might come after "be". We only care about the prediction for the very end of our sequence. The slicing `[:, -1, :]` selects all batches (the first `:`) and the logits corresponding to the *last* token (`-1`), giving us the probability distribution for the next token in the sequence.

5.  **`probs = F.softmax(last_token_logits, dim=-1)`**
    Logits are just raw scores. To turn them into probabilities that sum to 1, we apply the `softmax` function. Now we have a clean probability distribution.

6.  **`idx_next = torch.multinomial(probs, num_samples=1)`**
    Here's the sampling! Instead of just picking the token with the highest probability (which is called greedy decoding), `torch.multinomial` treats the `probs` tensor as a set of weights and randomly draws a sample from it. This is what gives the generation its flair and creativity.

7.  **`idx = torch.cat((idx, idx_next), dim=1)`**
    This is the "autoregressive" part. We append our newly sampled token (`idx_next`) to our running sequence (`idx`). In the next loop iteration, this slightly longer sequence will become the new input to the model.

In [ ]:
# --- Demonstration ---

# 1. Prepare our vocabulary and tokenizer
corpus = "hello world, this is a simple character-level gpt."
tokenizer = CharTokenizer(corpus)
vocab_size = tokenizer.vocab_size
print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {''.join(tokenizer.chars)}")

# 2. Create a mock model (with random, untrained weights)
model = MockGPT(vocab_size)

# 3. Define a starting prompt
prompt_string = "hello"
print(f"\nStarting prompt: '{prompt_string}'")

# 4. Encode the prompt into token indices
prompt_indices = tokenizer.encode(prompt_string)

# 5. Generate new tokens
generated_indices = generate(
    model=model,
    prompt_indices=prompt_indices,
    max_new_tokens=50
)

# 6. Decode the generated indices back to a string
generated_text = tokenizer.decode(generated_indices)
print("\n--- Generated Text ---")
print(generated_text)
print("------------------------")
print("\nNote: The output is random gibberish because our model is completely untrained.")
print("The key takeaway is that the *process* of generation is working correctly.")

### Going Deeper: The Context Window

In our simple `generate` function, the input sequence `idx` grows longer with every step. This seems fine, but it hides a critical limitation of Transformer models: the **context window** (often called `block_size` or `n_ctx`).

A Transformer can't look at an infinitely long sequence of past tokens. It has a fixed-size attention window, say 1024 tokens. If you give it a sequence of 1025 tokens, it will crash. Our current implementation will fail if we try to generate more tokens than the model's `block_size`.

How does the real `nanoGPT` handle this? It's a clever and efficient fix. At each step, it ensures the input to the model is never longer than the context window by truncating it. It only feeds the *last `block_size` tokens* of the sequence into the model.

Why does this work? The model only needs the most recent context to predict the next token. It doesn't need to re-read "Once upon a time" on page 1 to decide the next word on page 50. This keeps the generation process fast and prevents it from running out of memory or crashing, effectively giving the model a sliding "short-term memory."

Let's implement this more robust version.

In [ ]:
def generate_with_context_window(model, prompt_indices, max_new_tokens, block_size):
    """
    Generates text like before, but now respects the model's context window (block_size).
    """
    idx = torch.tensor(prompt_indices, dtype=torch.long).unsqueeze(0)

    for _ in range(max_new_tokens):
        # --- The Key Difference ---
        # If the sequence gets too long, crop it to the last `block_size` tokens.
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        # -------------------------

        # Get the logits using the (potentially cropped) sequence
        logits = model(idx_cond)
        # Focus on the last time step
        last_token_logits = logits[:, -1, :]
        # Apply softmax to get probabilities
        probs = F.softmax(last_token_logits, dim=-1)
        # Sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)
        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)

    return idx[0].tolist()

# --- Visualization with ASCII Art ---

# Let's imagine a block_size of 5 and trace the process.
block_size = 5
prompt = [1, 2, 3] # Represents an encoded prompt "abc"
print("--- Visualizing Generation with a Context Window ---")
print(f"Block size: {block_size}")
print(f"Initial prompt: {prompt}\n")

# Let's manually trace a few steps to see what `idx_cond` looks like.
idx = torch.tensor([prompt]) # Shape: (1, 3)

# Step 1
idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
print(f"Step 1: Sequence length is {idx.size(1)}. Input to model (idx_cond): {idx_cond.tolist()}")
idx = torch.cat((idx, torch.tensor([[4]])), dim=1) # Pretend we generated token '4'

# Step 2
idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
print(f"Step 2: Sequence length is {idx.size(1)}. Input to model (idx_cond): {idx_cond.tolist()}")
idx = torch.cat((idx, torch.tensor([[5]])), dim=1) # Pretend we generated token '5'

# Step 3 (Sequence length now equals block_size)
idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
print(f"Step 3: Sequence length is {idx.size(1)}. Input to model (idx_cond): {idx_cond.tolist()}")
idx = torch.cat((idx, torch.tensor([[6]])), dim=1) # Pretend we generated token '6'

# Step 4 (Sequence length is now > block_size)
idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
print(f"Step 4: Sequence length is {idx.size(1)}. Input to model (idx_cond): {idx_cond.tolist()} <-- TRUNCATED!")

# Step 5
idx = torch.cat((idx, torch.tensor([[7]])), dim=1) # Pretend we generated token '7'
idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
print(f"Step 5: Sequence length is {idx.size(1)}. Input to model (idx_cond): {idx_cond.tolist()} <-- TRUNCATED!")

### Summary

Congratulations! You've reached the end of our journey through the core components of nanoGPT. You've seen how text is turned into numbers, how a Transformer model learns from it, and now, how that model can use its knowledge to generate brand new text.

**What We Built:**
We created a `generate` function that implements the fundamental autoregressive sampling loop. We fed a prompt to a model, predicted the next token, sampled from the probabilities, and appended the result to our sequence, repeating the process to write text.

**Key Takeaways:**
-   **Autoregressive Loop:** Text generation is an iterative process where the model's output becomes its next input.
-   **Prediction vs. Sampling:** The model *predicts* a probability distribution, but we *sample* from it to create varied and interesting text.
-   **The Last Token Matters:** When generating, we only care about the model's prediction for the token that comes after the very end of our current sequence.
-   **Context is Finite:** Transformers have a limited memory (`block_size`), and a robust generation function must manage this by feeding only the most recent tokens back into the model.

**What's Next:**
This notebook marks the end of our simplified, from-scratch exploration. You now have the mental model to understand the real `nanoGPT` repository and other Transformer-based projects. I encourage you to dive into the official codebase, see how these concepts are implemented with configuration files, GPU acceleration, and other production-level details, and start training your own models